# Time Series Analysis in Medicine and Biology
## Practical Course — University of Tübingen · PfeiferLab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamsaraE/time-series-medicine-biology/blob/main/teaching/01_uk_respiratory_deaths_structural_model.ipynb)

---

# Notebook 01 — A Structural Model of UK Respiratory Deaths (1974–1979)

**In this notebook you will:**
1. Decompose a monthly mortality series into **trend + seasonal + residual** (additive model).
2. Build an explicit **structural generative model** of the series:

$$x_t = \underbrace{\text{Baseline} + \text{Trend}_t}_{\text{long-term level}} + \underbrace{\text{Seasonal}_t}_{\text{annual cycle}} + \underbrace{\text{Anomaly}_t}_{\text{unusual months}} + \underbrace{\text{Noise}_t}_{\text{random variation}}$$

3. **Detect anomalies** using a z-score on the residuals.
4. **Simulate** a realistic synthetic series from the estimated components.

**Dataset:** monthly deaths from bronchitis, emphysema, and asthma in the UK, 1974–1979, split by sex. A classic teaching series (from R's `datasets`), small enough to reason about by eye.

**Why this matters:** mortality and disease-incidence series almost always combine a slow trend, a strong annual cycle, and occasional spikes (e.g. a severe flu winter). Separating these is the first step in almost any epidemiological time-series analysis.

## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

## 2 · Load the data

We load the male and female series directly from a public mirror of R's datasets, so the
notebook runs anywhere (including Colab) with no local files. Each series is monthly; we
attach a proper monthly `DatetimeIndex` starting in January 1974.

In [ ]:
fdeaths_url = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/fdeaths.csv"
mdeaths_url = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/mdeaths.csv"

fdeaths = pd.read_csv(fdeaths_url, index_col=0)
mdeaths = pd.read_csv(mdeaths_url, index_col=0)

female = fdeaths["value"]
male = mdeaths["value"]

# Attach a monthly datetime index (ME = month-end; replaces the deprecated "M")
dates = pd.date_range(start="1974-01-01", periods=len(male), freq="ME")
male.index = dates
female.index = dates

print(f"{len(male)} monthly observations, {male.index.min().date()} to {male.index.max().date()}")
male.head()

## 3 · Look at the raw series first

Always plot before modelling. Two questions to ask any new series:
- **Is there a trend?** (a slow drift up or down)
- **Is there seasonality, and is its size constant?** (constant amplitude → *additive*; growing amplitude → *multiplicative*)

Here both series show a clear yearly cycle — deaths peak in winter — with roughly constant
amplitude, which is what justifies the **additive** decomposition we use next.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(male, color="steelblue")
ax[0].set_title("Male respiratory mortality")
ax[0].set_ylabel("Deaths / month")
ax[1].plot(female, color="indianred")
ax[1].set_title("Female respiratory mortality")
ax[1].set_ylabel("Deaths / month")
ax[1].set_xlabel("Year")
plt.tight_layout()
plt.show()

## 4 · Additive decomposition

`seasonal_decompose` with `model="additive"` and `period=12` splits each series into
**trend**, **seasonal**, and **residual** components that *add back* to the original.
We use it here as a quick, standard baseline before building our own structural model.

In [ ]:
decomp_male = seasonal_decompose(male, model="additive", period=12)
decomp_male.plot()
plt.suptitle("Additive decomposition — male", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
decomp_female = seasonal_decompose(female, model="additive", period=12)
decomp_female.plot()
plt.suptitle("Additive decomposition — female", y=1.02)
plt.tight_layout()
plt.show()

## 5 · A structural estimation function

Instead of relying only on the black-box decomposition, we now build the components
**explicitly**, so every term in the model equation is something we computed ourselves:

- **Baseline + trend** — a straight line fit by least squares (`np.polyfit`, degree 1).
- **Seasonal** — the average deviation of each calendar month from the overall mean.
- **Residual** — whatever is left after removing trend and season.
- **Noise level** — the standard deviation of that residual.

This makes the generative equation at the top of the notebook concrete and reusable.

In [ ]:
def estimate_structure(series):
    """Decompose a monthly series into baseline+trend, seasonal, residual, and a noise level."""
    t = np.arange(len(series))

    # Baseline + linear trend (degree-1 least squares fit)
    coeffs = np.polyfit(t, series.values, 1)
    baseline_trend = coeffs[0] * t + coeffs[1]

    # Seasonal pattern: mean deviation of each calendar month
    monthly_mean = series.groupby(series.index.month).mean()
    seasonal_pattern = monthly_mean - monthly_mean.mean()
    season = np.tile(seasonal_pattern.values, len(series) // 12)

    # Structural residual and its spread
    residual = series.values - (baseline_trend + season)
    noise_std = residual.std()

    return baseline_trend, season, residual, noise_std

In [ ]:
baseline_trend_m, season_m, residual_m, noise_std_m = estimate_structure(male)
baseline_trend_f, season_f, residual_f, noise_std_f = estimate_structure(female)

print(f"Male   noise std: {noise_std_m:6.1f}")
print(f"Female noise std: {noise_std_f:6.1f}")

## 6 · Anomaly detection

We flag a month as **anomalous** when its residual is more than 2 standard deviations from
zero — i.e. an absolute z-score above 2:

$$z_t = \frac{\text{residual}_t}{\sigma}, \qquad \text{anomaly if } |z_t| > 2$$

These are months whose mortality is far higher (or lower) than the trend + season alone
would predict — often severe-winter or epidemic months.

In [ ]:
z_m = residual_m / noise_std_m
anomaly_idx_m = np.where(np.abs(z_m) > 2)[0]

z_f = residual_f / noise_std_f
anomaly_idx_f = np.where(np.abs(z_f) > 2)[0]

print(f"Male anomalies:   {len(anomaly_idx_m)}  at {list(male.index[anomaly_idx_m].strftime('%Y-%m'))}")
print(f"Female anomalies: {len(anomaly_idx_f)}  at {list(female.index[anomaly_idx_f].strftime('%Y-%m'))}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(male, color="steelblue", label="observed")
ax[0].scatter(male.index[anomaly_idx_m], male.iloc[anomaly_idx_m],
              color="red", zorder=5, label="anomaly (|z| > 2)")
ax[0].set_title("Male — anomalies"); ax[0].set_ylabel("Deaths"); ax[0].legend()
ax[1].plot(female, color="indianred", label="observed")
ax[1].scatter(female.index[anomaly_idx_f], female.iloc[anomaly_idx_f],
              color="red", zorder=5, label="anomaly (|z| > 2)")
ax[1].set_title("Female — anomalies"); ax[1].set_ylabel("Deaths"); ax[1].set_xlabel("Year"); ax[1].legend()
plt.tight_layout()
plt.show()

## 7 · Simulating a synthetic series

Because we have every structural component, we can **generate** a new realistic series by
adding them back together and replacing the random part with fresh Gaussian noise:

$$\hat{x}_t = \text{Baseline}_t + \text{Trend}_t + \text{Seasonal}_t + \text{Anomaly}_t + \varepsilon_t,\qquad \varepsilon_t \sim \mathcal{N}(0, \sigma^2)$$

We keep the detected anomalies fixed and draw new noise, so each run produces a plausible
*alternative* history with the same trend, seasonality, and variability. The random seed
makes the result reproducible.

In [ ]:
def simulate(series, baseline_trend, season, residual, noise_std, seed=42):
    """Generate a synthetic series from the estimated structural components."""
    z = residual / noise_std
    anomaly_idx = np.where(np.abs(z) > 2)[0]

    anomaly_component = np.zeros(len(series))
    anomaly_component[anomaly_idx] = residual[anomaly_idx]

    rng = np.random.default_rng(seed)
    noise = rng.normal(0, noise_std, len(series))

    return baseline_trend + season + anomaly_component + noise

In [ ]:
sim_male = simulate(male, baseline_trend_m, season_m, residual_m, noise_std_m)
sim_female = simulate(female, baseline_trend_f, season_f, residual_f, noise_std_f)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(male.index, male.values, label="observed", color="steelblue")
ax[0].plot(male.index, sim_male, label="simulated", color="orange", alpha=0.8)
ax[0].set_title("Male — observed vs simulated"); ax[0].set_ylabel("Deaths"); ax[0].legend()
ax[1].plot(female.index, female.values, label="observed", color="indianred")
ax[1].plot(female.index, sim_female, label="simulated", color="orange", alpha=0.8)
ax[1].set_title("Female — observed vs simulated"); ax[1].set_ylabel("Deaths"); ax[1].set_xlabel("Year"); ax[1].legend()
plt.tight_layout()
plt.show()

## Key takeaways

- A monthly mortality series can be read as **trend + seasonal + residual**; plotting first tells you whether an *additive* model is appropriate (constant seasonal amplitude).
- Building the components **explicitly** (line fit + monthly means) makes the generative model transparent and lets you simulate new data.
- A simple **z-score on the residual** is enough to flag unusual months; in the next notebooks we compare this with more robust alternatives (MAD).

**Try it yourself:**
1. Change the anomaly threshold from 2 to 1.5 or 3. How does the count change?
2. Replace the linear trend with a quadratic (`np.polyfit(..., 2)`). Does the residual shrink?
3. Run `simulate(...)` with several different seeds and overlay the results.